In [8]:
from langchain.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from config import LLAMA_MODEL
from langchain_community.vectorstores import FAISS
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_core.pydantic_v1 import BaseModel,Field

In [60]:
retriever = FAISS.load_local('web_data',OllamaEmbeddings(model=LLAMA_MODEL),allow_dangerous_deserialization=True).as_retriever()

In [61]:
response = retriever.invoke('what is langsmith?')

In [63]:
retriever.get_relevant_documents('what is langsmith?')

C:\Users\umend\AppData\Local\Temp\ipykernel_12492\2677745439.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  retriever.get_relevant_documents('what is langsmith?')


[Document(metadata={'source': 'https://docs.smith.langchain.com/overview', 'title': 'Get started with LangSmith | 🦜️🛠️ LangSmith', 'description': 'LangSmith is a platform for building production-grade LLM applications.', 'language': 'en'}, page_content='Learn more about tracing in the observability tutorials, conceptual guide and how-to guides.\n5. View your trace\u200b\nBy default, the trace will be logged to the project with the name default. You should see the following sample output trace logged using the above code.\n6. Run your first evaluation\u200b\nEvaluations help assess application performance by testing the application against a given set of inputs. Evaluations require a system to test, data to serve as test cases, and evaluators to grade the results.\nHere we are running an evaluation against a sample dataset using a simple custom evaluator that checks if the real output exactly matches our gold-standard output.'),
 Document(metadata={'source': 'https://docs.smith.langchai

In [13]:
llm = ChatOllama(model='llama3.2')

class Data(BaseModel):
    binary_score:str = Field(description="Documents are relevant to question, 'yes' or 'no' ")

grader = llm.with_structured_output(Data)

prompt = """ You are a grader accessing relevance of a retreived document to a user questin. \n
           If a document contains keyword(s) or semanticmeaning related to question, grade it as relevant. \n
           Give a binary score 'yes' or 'no' score to indicate whether document is relevant to question"""

template = ChatPromptTemplate.from_messages(
   [ ('system',prompt),
    ("human","Retrieved document: \n\n {document} \n\n User question: {question}"),]
)

model = template | grader

In [35]:
response

[Document(metadata={'source': 'https://docs.smith.langchain.com/overview', 'title': 'Get started with LangSmith | 🦜️🛠️ LangSmith', 'description': 'LangSmith is a platform for building production-grade LLM applications.', 'language': 'en'}, page_content='Learn more about tracing in the observability tutorials, conceptual guide and how-to guides.\n5. View your trace\u200b\nBy default, the trace will be logged to the project with the name default. You should see the following sample output trace logged using the above code.\n6. Run your first evaluation\u200b\nEvaluations help assess application performance by testing the application against a given set of inputs. Evaluations require a system to test, data to serve as test cases, and evaluators to grade the results.\nHere we are running an evaluation against a sample dataset using a simple custom evaluator that checks if the real output exactly matches our gold-standard output.'),
 Document(metadata={'source': 'https://docs.smith.langchai

In [17]:
model.invoke({'question':'what is langsmith?', 'document':response[0].page_content})

Data(binary_score='no')

In [54]:
prompt =  ''' Your task is to answer the Question based on following context:
                   {context}

                   Question : {question}
               '''

prompt_template = ChatPromptTemplate.from_template(template=prompt)

retriever_model = (
    prompt_template | ChatOllama(model='llama3.2')
)

In [57]:
import json
from fastapi.encoders import jsonable_encoder

res=retriever_model.invoke({'context':response,'question':'what is langsmith?'})

In [15]:
res

NameError: name 'res' is not defined

In [ ]:
val = model.invoke({'question':'what is langsmith?','document':response[0].page_content})

In [21]:
val.binary_score

'no'

In [ ]:
ls = []

for d in r:
    ls.append(d.page_content)



SELF Correctiev RAG

In [1]:
from langchain_ollama import ChatOllama
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from langchain_core.runnables import RunnablePassthrough
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.vectorstores import FAISS
from config import LLAMA_MODEL
from typing_extensions import TypedDict
from typing import List
import os
from langchain_core.pydantic_v1 import BaseModel,Field
from langchain.schema import Document

In [2]:
retriever = FAISS.load_local('web_data',OllamaEmbeddings(model=LLAMA_MODEL),allow_dangerous_deserialization=True).as_retriever()
llm_model = ChatOllama(model='llama3.2')

os.environ["TAVILY_API_KEY"] = "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"
web_search_tool = TavilySearchResults(max_results=4)

In [3]:
class structured_output(BaseModel):
    binary_data: str= Field(description="Documents are relevant to question, 'yes' or 'no' ")

llm = llm_model.with_structured_output(structured_output)

prompt = """ You are a grader accessing relevance of a retreived document to a user questin. \n
           If a document contains keyword(s) or semanticmeaning related to question, grade it as relevant. \n
           Give a binary score 'yes' or 'no' score to indicate whether document is relevant to question"""

template = ChatPromptTemplate.from_messages(
   [ ('system',prompt),
    ("human","Retrieved document: \n\n {document} \n\n User question: {question}"),]
)

grader = template | llm

In [49]:
question_rewrite_prompt = '''You are a question re-writer that converts input question to a better version that must be optimized \n 
                           for a web search. Only write new question not other things.'''

question_rewrite_template = ChatPromptTemplate.from_messages(
    [
        ('system',question_rewrite_prompt),
        ('human','Here is the initial question:\n\n {question} \n formulate question into improved version which must be short and specific.')
    ]
)

rewrite_model = question_rewrite_template | llm_model

In [50]:
rewrite_model.invoke({'question':'what is langsmith?'})

AIMessage(content='Improved question: What is Lang-Smooth, also known as LangSmith, in language processing or NLP?', response_metadata={'model': 'llama3.2', 'created_at': '2024-12-28T19:49:18.7179272Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 6175355900, 'load_duration': 64061800, 'prompt_eval_count': 83, 'prompt_eval_duration': 203000000, 'eval_count': 24, 'eval_duration': 5901000000}, id='run-98029c91-fd18-421f-8b45-18b42e120093-0', usage_metadata={'input_tokens': 83, 'output_tokens': 24, 'total_tokens': 107})

In [51]:
class AgentState(TypedDict):
    question:str
    generation:str
    web_search:str
    documents:List[Document]

In [57]:
def retreiver_document(state):
    print("In Retriever")
    question = state['question'] 
    
    docs = retriever.get_relevant_documents(question)   #response = retriever.invoke('what is langsmith?')
    print(docs)
    return {'documents':docs,'question':question}

def generator(state):
    print("\n In Generator \n")
    question = state['question']
    docs = state['documents']

    generator_prompt =  ''' Your task is to provide answer for the question based on following context:
                   {context}

                   Question : {question}
               '''
    generator_template = ChatPromptTemplate.from_template(template=generator_prompt)
    generator_model = generator_template | llm_model

    data = generator_model.invoke({'context':docs,'question':question})

    return {'documents':docs,'question':question,'generation':data.content}

def grader_function(state):
    print("\n In Grader \n")
    docs = state['documents']
    question = state['question']

    web_search_required = 'yes'
    valid_docs = []
    for response in docs:
        value = grader.invoke({'question':question,'document':response.page_content})
        print(value)
        if value.binary_data=='yes':
            valid_docs.append(response)
        else:
            web_search_required='no'
   
    return {'documents':valid_docs,'web_search':web_search_required,'question':question}

def router(state):
    print("\n In Router \n")
    web_search_required = state['web_search']
    if web_search_required=='no':
        return 'transform_query'
    else:
        return 'generator'
    
def web_search_function(state):
    print("\n In web Search \n")
    question = state['question']
    docs = state['documents']
    
    print(state)

    url = web_search_tool.invoke({'query':question})
    print("\nURL\n",url)
    web_result = '\n'.join([d['content'] for d in url])
    new_doc = Document(page_content=web_result)
    docs.append(new_doc)
    return {'question':question,'documents':docs}

def transform_query(state):
    print('\n In Transform \n')
    question = state['question']
    new_question = rewrite_model.invoke({'question':question})
    return {'documents':state['documents'],'question':new_question.content}

In [58]:
workflow = StateGraph(AgentState)

workflow.add_node('retriever',retreiver_document)
workflow.add_node('generator',generator)
workflow.add_node('grader',grader_function)
workflow.add_node('transform_query',transform_query)
workflow.add_node('web_search_function',web_search_function)

workflow.set_entry_point('retriever')

workflow.add_edge('retriever','grader')

workflow.add_conditional_edges('grader',router,{'transform_query':'transform_query','generator':'generator'})

workflow.add_edge('transform_query','web_search_function')
workflow.add_edge('web_search_function','generator')
workflow.add_edge('generator',END)

app = workflow.compile()

In [59]:
state = {'question':'what is langsmith?'}
response = app.invoke(state)


In Retriever
[Document(metadata={'source': 'https://docs.smith.langchain.com/overview', 'title': 'Get started with LangSmith | 🦜️🛠️ LangSmith', 'description': 'LangSmith is a platform for building production-grade LLM applications.', 'language': 'en'}, page_content='Learn more about tracing in the observability tutorials, conceptual guide and how-to guides.\n5. View your trace\u200b\nBy default, the trace will be logged to the project with the name default. You should see the following sample output trace logged using the above code.\n6. Run your first evaluation\u200b\nEvaluations help assess application performance by testing the application against a given set of inputs. Evaluations require a system to test, data to serve as test cases, and evaluators to grade the results.\nHere we are running an evaluation against a sample dataset using a simple custom evaluator that checks if the real output exactly matches our gold-standard output.'), Document(metadata={'source': 'https://docs.sm

In [60]:
response

{'question': 'Improved version:\n"What is Langsmith, a language processing tool?"',
 'generation': 'Here\'s an improved version of the answer:\n\n"Langsmith is an application framework for building production-grade LLM applications. It provides a comprehensive suite of tools and features to help take language models from prototype to production, offering advanced debugging and orchestration capabilities tailored to managing complex AI systems in production."',
 'web_search': 'no',
 'documents': [Document(metadata={'source': 'https://docs.smith.langchain.com/overview', 'title': 'Get started with LangSmith | 🦜️🛠️ LangSmith', 'description': 'LangSmith is a platform for building production-grade LLM applications.', 'language': 'en'}, page_content='Get started with LangSmith | 🦜️🛠️ LangSmith\n\n\n\n\n\n\nSkip to main contentLearn the essentials of LangSmith in the new Introduction to LangSmith course!  Enroll for free. API ReferenceRESTPythonSearchRegionUSEUGo to AppQuick StartObservability

In [56]:
len(response['documents'])

2